In [2]:

import os
import re
import pandas as pd

PROJECT_DIR = r"D:\MCA58-Tanya-Singh-AI-Food-Recommendation"

ORIGINAL_FILE = os.path.join(
    PROJECT_DIR, "data", "original", "Food_Dataset.xlsx"
)

API_DIR = os.path.join(
    PROJECT_DIR, "data", "api_data"
)


def normalize_food_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower().strip()
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name)

    return name


# Load original Excel
original_df = pd.read_excel(ORIGINAL_FILE)

original_df["match_key"] = (
    original_df["food_name"].apply(normalize_food_name)
)


def prepare_api_for_fusion(df, source_name):
    if df.empty:
        return pd.DataFrame(columns=["match_key"])

    if "query_food" not in df.columns:
        print(source_name, "does not contain query_food.")
        return pd.DataFrame(columns=["match_key"])

    df = df.copy()

    df["match_key"] = df["query_food"].apply(
        normalize_food_name
    )

    df = df[df["match_key"] != ""]

    def combine_values(series):
        values = series.dropna().astype(str).str.strip()
        values = values[values != ""].drop_duplicates()
        return " || ".join(values)

    columns = [
        col for col in df.columns
        if col not in ["query_food", "match_key"]
    ]

    grouped = (
        df.groupby("match_key")[columns]
        .agg(combine_values)
        .reset_index()
    )

    grouped = grouped.rename(
        columns={
            col: f"{source_name}_{col}"
            for col in columns
        }
    )

    return grouped


# Load collected API datasets
usda_df = pd.read_csv(
    os.path.join(API_DIR, "usda_data.csv")
)

spoonacular_df = pd.read_csv(
    os.path.join(API_DIR, "spoonacular_data.csv")
)

themealdb_df = pd.read_csv(
    os.path.join(API_DIR, "themealdb_data.csv")
)

weather_path = os.path.join(API_DIR, "weather_data.csv")

weather_df = (
    pd.read_csv(weather_path)
    if os.path.exists(weather_path)
    else pd.DataFrame()
)

# Prepare API data
usda_fusion = prepare_api_for_fusion(usda_df, "usda")

spoonacular_fusion = prepare_api_for_fusion(
    spoonacular_df, "spoonacular"
)

themealdb_fusion = prepare_api_for_fusion(
    themealdb_df, "themealdb"
)

# Merge API data with original dataset
final_fused_df = original_df.copy()

for api_df in [
    usda_fusion,
    spoonacular_fusion,
    themealdb_fusion
]:
    if not api_df.empty:
        final_fused_df = final_fused_df.merge(
            api_df,
            on="match_key",
            how="left",
            validate="many_to_one"
        )

# Remove temporary column
final_fused_df.drop(
    columns=["match_key"],
    inplace=True
)

print("Fusion recreated successfully!")
print("Rows:", len(final_fused_df))
print("Columns:", len(final_fused_df.columns))

display(final_fused_df.head())


Fusion recreated successfully!
Rows: 200
Columns: 46


,food_id,food_name,cuisine,meal_type,diet,ingredients,calories,protein_g,carbs_g,fat_g,...,themealdb_food_name,themealdb_meal_id,themealdb_category,themealdb_area,themealdb_ingredients,themealdb_instructions,themealdb_image_url,themealdb_source,themealdb_normalized_name,themealdb_normalized_query
0,F001,Paneer Tikka,Indian,Dinner,Vegetarian,Paneer;Onion;Capsicum;Yogurt;Spices,497,12,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,F002,Masala Dosa,Indian,Breakfast,Vegetarian,Rice;Urad Dal;Potato;Onion,472,32,14,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F003,Poha,Indian,Breakfast,Vegetarian,Flattened Rice;Onion;Peanuts;Curry Leaves,502,31,26,17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,F004,Chole,Indian,Lunch,Vegetarian,Chickpeas;Tomato;Onion;Spices,344,22,21,9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,F005,Chicken Tikka,Indian,Dinner,Non-Vegetarian,Chicken;Yogurt;Spices,346,21,63,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:

import os
import pandas as pd

# Main AI Food Recommendation project folder
PROJECT_DIR = r"D:\MCA58-Tanya-Singh-AI-Food-Recommendation"

# Final fused dataset
final_csv_path = os.path.join(
    PROJECT_DIR,
    "FINAL_FUSED_FOOD_DATASET.csv"
)

final_excel_path = os.path.join(
    PROJECT_DIR,
    "FINAL_FUSED_FOOD_DATASET.xlsx"
)

# Export CSV
final_fused_df.to_csv(
    final_csv_path,
    index=False,
    encoding="utf-8-sig"
)

# Export Excel with two worksheets
with pd.ExcelWriter(
    final_excel_path,
    engine="openpyxl"
) as writer:

    final_fused_df.to_excel(
        writer,
        sheet_name="Final Fused Dataset",
        index=False
    )

    summary = pd.DataFrame([
        {
            "API Source": source,
            "Matched Records": (
                final_fused_df[columns]
                .notna()
                .any(axis=1)
                .sum() if columns else 0
            )
        }
        for source, columns in sources.items()
    ])

    summary.to_excel(
        writer,
        sheet_name="Fusion Summary",
        index=False
    )

print("=" * 55)
print("FINAL FUSION FILES CREATED SUCCESSFULLY")
print("=" * 55)

print("\nCSV:", final_csv_path)
print("Excel:", final_excel_path)

print("\nTotal Records:", len(final_fused_df))
print("Total Attributes:", len(final_fused_df.columns))


NameError: name 'sources' is not defined